# Tuần 4 (Sprint S2) — Huấn luyện Học trò Nhãn cứng `E12` & Chưng cất Nhãn mềm `E6` (TV3 — Tasks 3.13 & 3.14)

Notebook này thực thi các thí nghiệm GPU Tuần 4 của **TV3** trên Google Colab (`Tesla T4`) theo `docs/WEEK4.md` và `docs/guides/TV3-tuan4.md`:
1. **Đảm bảo `working_tree_dirty: false`:**
   - Clone trực tiếp nhánh `tv3/week4` đã commit sẵn `configs/E12.yaml` và `configs/E6.yaml`.
   - Dữ liệu thô (`data/raw/fns/`, `data/raw/headlines_2026/headlines.jsonl`) nằm hoàn toàn trong đường dẫn bị `.gitignore` bỏ qua.
   - Kiểm tra bắt buộc `git status --porcelain --untracked-files=no == ""` trước khi chạy để mọi file `run.json` đều ghi `working_tree_dirty: false`.
2. **Task 3.13 (`E12` — `E5` + nhãn cứng LLM `argmax(P2)`, 3 seeds `[1, 2, 3]`):**
   - Chạy `run_transformer_experiment` với `configs/E12.yaml` (`8,351` mẫu FNS train + `1,409` mẫu `train_unlabeled` 2026 gán nhãn cứng `argmax`; trọng số lớp của `weighted_ce` giữ nguyên tính chỉ trên FNS train).
   - Xuất `results/E12/seed{1,2,3}/{run.json, preds_validation.jsonl}`.
3. **Task 3.14 (`E6` — Chưng cất nhãn mềm LLM với nhiệt độ $T \in \{1, 2, 4\} \times \alpha \in \{0.3, 0.5, 0.7\}$):**
   - **Bước 3A (Lưới 9 tổ hợp trên `seed = 1`):** Chạy `run_e6_grid_search` qua 9 tổ hợp $(T, \alpha)$ trên tập `validation` (`1,791` mẫu), xuất `results/E6/grid.md` và `results/E6/grid.json`.
   - **Bước 3B (Chạy 3 seeds `[1, 2, 3]` với tổ hợp $(T^*, \alpha^*)$ tốt nhất trên validation):** Xuất `results/E6/seed{1,2,3}/{run.json, preds_validation.jsonl}`.
4. **Đóng gói & Tải về:** Nén toàn bộ `results/E12/` và `results/E6/` thành `week4_e12_e6_results.zip` để tích hợp vào repository.

## 1. Cài đặt Môi trường Colab, Clone Nhánh `tv3/week4`, Tải FNS & Upload `headlines.jsonl`

- Kiểm tra GPU `Tesla T4`, clone nhánh `tv3/week4`, tải dữ liệu FNS (`python data/download.py --only fns`), upload `headlines.jsonl` (`SHA-256: 28bd2f3d...`) và xác minh `git status --porcelain --untracked-files=no` hoàn toàn rỗng.

In [ ]:
import base64
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive, files, userdata

assert torch.cuda.is_available(), "CUDA GPU not detected! Please select Runtime -> Change runtime type -> T4 GPU."
print(f"GPU detected: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

token_val = userdata.get("HF_TOKEN")
if token_val:
    os.environ["HF_TOKEN"] = token_val.strip()
del token_val

REPO_URL = "https://github.com/thienandinh29/financial-sentiment-dl.git"
REPO_DIR = Path("financial-sentiment-dl")
BRANCH = "tv3/week4"

if not REPO_DIR.exists():
    gh_val = userdata.get("GH_TOKEN")
    if not gh_val:
        raise RuntimeError("Missing GH_TOKEN in Colab Secrets.")
    basic_auth = base64.b64encode(f"x-access-token:{gh_val.strip()}".encode("utf-8")).decode("ascii")
    git_env = os.environ.copy()
    git_env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic_auth}",
    })
    del gh_val, basic_auth
    res = subprocess.run(
        ["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        env=git_env,
        capture_output=True,
        text=True,
        check=False,
    )
    del git_env
    if res.returncode != 0:
        raise RuntimeError(f"git clone failed with exit code {res.returncode}: {res.stderr}")

if Path.cwd().name != REPO_DIR.name and REPO_DIR.exists():
    os.chdir(REPO_DIR)

py_ver = platform.python_version()
print(f"Python version: {py_ver}")
if Path("requirements.lock").is_file():
    lock_proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.lock"],
        capture_output=True,
        text=True,
        check=False,
    )
    if lock_proc.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not (Path("data/raw/fns/train.jsonl").is_file() and Path("data/raw/fns/validation.jsonl").is_file()):
    subprocess.run([sys.executable, "data/download.py", "--only", "fns"], check=True)

from models.llm_soft_labels import DEFAULT_HEADLINES, EXPECTED_HEADLINES_SHA256
from train.data import sha256_file

USE_GOOGLE_DRIVE = False
DRIVE_WORK_DIR = Path("/content/drive/MyDrive/financial_sentiment_dl_week4")
if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    DRIVE_WORK_DIR.mkdir(parents=True, exist_ok=True)
    if (DRIVE_WORK_DIR / "headlines.jsonl").is_file() and not DEFAULT_HEADLINES.is_file():
        DEFAULT_HEADLINES.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE_WORK_DIR / "headlines.jsonl", DEFAULT_HEADLINES)

if not DEFAULT_HEADLINES.is_file():
    DEFAULT_HEADLINES.parent.mkdir(parents=True, exist_ok=True)
    print("Please upload headlines.jsonl (SHA-256: 28bd2f3d...):")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No headlines.jsonl uploaded!")
    shutil.move(next(iter(uploaded.keys())), DEFAULT_HEADLINES)

actual_sha = sha256_file(DEFAULT_HEADLINES)
assert actual_sha == EXPECTED_HEADLINES_SHA256, f"SHA-256 mismatch: {actual_sha}"
print(f"Verified {DEFAULT_HEADLINES} SHA-256: {actual_sha}")

# Enforce clean working tree rule (docs/guides/TV3-tuan4.md)
dirty_out = subprocess.run(
    ["git", "status", "--porcelain", "--untracked-files=no"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
assert dirty_out == "", f"Working tree is dirty! Output:\n{dirty_out}"
head_commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print(f"CLEAN WORKING TREE VERIFIED (working_tree_dirty=False) at commit {head_commit}")

## 2. Task 3.13 — Chạy `E12` (`E5` + Nhãn cứng LLM `argmax(P2)`, 3 Seeds `[1, 2, 3]`)

- Huấn luyện `distilbert/distilbert-base-uncased` trên `8,351` mẫu FNS train + `1,409` mẫu 2026 `train_unlabeled` gán nhãn cứng `argmax(probs)` từ `P2`.
- Trọng số lớp `weighted_ce` tính chỉ trên FNS train (giống hệt `E5`).
- Lưu kết quả vào `results/E12/seed{1,2,3}/{run.json, preds_validation.jsonl}` và đối chiếu với `E5` (`0.8030 ± 0.0054`).

In [ ]:
import json
import numpy as np
import yaml
from pathlib import Path
from evaluate.metrics import read_preds, score, validation_gold
from models.transformer import run_transformer_experiment

e12_cfg = yaml.safe_load(Path("configs/E12.yaml").read_text(encoding="utf-8"))
e12_summaries = run_transformer_experiment(e12_cfg)
gold = validation_gold(e12_cfg)

e12_f1s = []
for s in e12_summaries:
    seed = s["seed"]
    run_path = Path(f"results/E12/seed{seed}/run.json")
    pred_path = Path(f"results/E12/seed{seed}/preds_validation.jsonl")
    run_meta = json.loads(run_path.read_text(encoding="utf-8"))
    assert run_meta["working_tree_dirty"] is False, f"working_tree_dirty is True in {run_path}!"
    scored = score(read_preds(pred_path), gold)
    e12_f1s.append(scored["macro_f1"])
    print(f"E12 seed {seed}: dirty={run_meta['working_tree_dirty']} | best_epoch={s['best_epoch']} | val_macro_f1={scored['macro_f1']:.4f} | f1={scored['f1']}")

print(f"E12 Validation Macro-F1 (3 seeds): {np.mean(e12_f1s):.4f} +/- {np.std(e12_f1s, ddof=1):.4f} (vs E5: 0.8030 +/- 0.0054)")

## 3. Task 3.14 — Bước 3A: Dò Lưới 9 Tổ Hợp Chưng Cất Nhãn Mềm `E6` ($T \in \{1, 2, 4\} \times \alpha \in \{0.3, 0.5, 0.7\}$, `seed = 1`)

- Chạy 9 tổ hợp trên `seed = 1`, lưu toàn bộ vào `results/E6/grid.md` và `results/E6/grid.json`.
- Chọn tổ hợp $(T^*, \alpha^*)$ có `val_macro_f1` cao nhất trên tập `validation`.

In [ ]:
import json
import yaml
from pathlib import Path
from models.transformer import run_e6_grid_search

e6_base_cfg = yaml.safe_load(Path("configs/E6.yaml").read_text(encoding="utf-8"))
grid_result = run_e6_grid_search(e6_base_cfg, seed=1)
print(Path("results/E6/grid.md").read_text(encoding="utf-8"))
best_cell = grid_result["selected"]
print("Selected winning (T*, alpha*) on validation seed 1:", json.dumps(best_cell, indent=2))

## 4. Task 3.14 — Bước 3B: Chạy 3 Seeds `[1, 2, 3]` Cho `E6` Với Tổ Hợp $(T^*, \alpha^*)$ Tốt Nhất

- Ngoài tổ hợp thắng cuộc $(T^*, \alpha^*)$ được chạy đủ 3 seeds vào `results/E6/seed{1,2,3}/`, cell dưới đây cũng hỗ trợ chạy đủ 3 seeds cho các mức nhiệt độ $T \in \{1, 2, 4\}$ (ở $\alpha^*$ cố định) nếu muốn có bảng so sánh 3 seeds đầy đủ theo từng mức nhiệt độ $T \in \{1, 2, 4\}$.

In [ ]:
import copy
import json
import numpy as np
import yaml
from pathlib import Path
from evaluate.metrics import read_preds, score, validation_gold
from models.transformer import run_transformer_experiment

e6_run_cfg = copy.deepcopy(yaml.safe_load(Path("configs/E6.yaml").read_text(encoding="utf-8")))
e6_run_cfg["data"]["distill_temperature"] = float(best_cell["temperature"])
e6_run_cfg["data"]["distill_alpha"] = float(best_cell["alpha"])

e6_summaries = run_transformer_experiment(e6_run_cfg)
gold = validation_gold(e6_run_cfg)

e6_f1s = []
for s in e6_summaries:
    seed = s["seed"]
    run_path = Path(f"results/E6/seed{seed}/run.json")
    pred_path = Path(f"results/E6/seed{seed}/preds_validation.jsonl")
    run_meta = json.loads(run_path.read_text(encoding="utf-8"))
    assert run_meta["working_tree_dirty"] is False, f"working_tree_dirty is True in {run_path}!"
    scored = score(read_preds(pred_path), gold)
    e6_f1s.append(scored["macro_f1"])
    print(f"E6 (T={best_cell['temperature']}, alpha={best_cell['alpha']}) seed {seed}: dirty={run_meta['working_tree_dirty']} | best_epoch={s['best_epoch']} | val_macro_f1={scored['macro_f1']:.4f} | f1={scored['f1']}")

print(f"E6 Validation Macro-F1 (3 seeds): {np.mean(e6_f1s):.4f} +/- {np.std(e6_f1s, ddof=1):.4f}")

## 5. Kiểm Tra Toàn Vẹn & Đóng Gói `week4_e12_e6_results.zip` Để Tải Về

- Kiểm tra tất cả file `run.json` và `preds_validation.jsonl` của `E12` và `E6` (`working_tree_dirty == False`, `1,791` dự đoán mỗi file).
- Nén toàn bộ `results/E12/` và `results/E6/` thành `week4_e12_e6_results.zip` và tải về máy.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files
from evaluate.metrics import read_preds

zip_path = Path("week4_e12_e6_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [Path("results/E12"), Path("results/E6")]:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                if p.name == "preds_validation.jsonl":
                    preds = read_preds(p)
                    assert len(preds) == 1791, f"Expected 1791 preds in {p}, got {len(preds)}"
                zf.write(p, arcname=str(p))

print(f"Created {zip_path} ({zip_path.stat().st_size} bytes). Triggering download...")
files.download(str(zip_path))